# 01. Baseline: GARCH(1,1)-t на реальном train

Эталон (B1). Обучаем GARCH(1,1) с t-распределением остатков на реальном `train` (2010–2019), затем фиксируем параметры (μ, ω, α, β, ν) и делаем one-step-ahead прогноз волатильности на каждый день `test` (2020–2024).

Формула рекурсии при фиксированных параметрах:

\[ \sigma_t^2 = \omega + \alpha (r_{t-1}-\mu)^2 + \beta\,\sigma_{t-1}^2 \]

Это **fixed-parameter walk-forward**: GARCH «переиспользует» свою же формулу на новых данных, без переоптимизации параметров. Такой режим даёт чистое сравнение **«какие данные оставили какой след»** в обученной модели.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from src import data as dm, garch_eval as ge

prices, returns = dm.load_or_download()
train, test = dm.train_test_split(returns)

params, res = ge.fit_garch_const_mean(train, dist="t")
print("Параметры GARCH(1,1)-t на train:")
print(json.dumps(params.to_dict(), indent=2, ensure_ascii=False))
print(f"\npersistence = α + β = {params.alpha + params.beta:.4f}")

Параметры GARCH(1,1)-t на train:
{
  "mu": 0.08679058700313227,
  "omega": 0.02529362938666317,
  "alpha": 0.175878303419592,
  "beta": 0.8125569611160142,
  "nu": 4.873853855508748,
  "dist": "t"
}

persistence = α + β = 0.9884


In [2]:
df_b1 = ge.walk_forward_fixed(params, history=train, test=test)
metrics_b1 = ge.evaluate_forecast(df_b1, params)
display(df_b1.head())
pd.Series(metrics_b1).to_frame("B1").T

,r_pct,sigma_pct
date,,
2020-01-02,0.834390,0.519803
2020-01-03,-0.708491,0.585783
2020-01-06,0.352715,0.644480
2020-01-07,-0.280718,0.612560
2020-01-08,0.489047,0.594932


,n,RMSE_abs_r,MAE_abs_r,QLIKE,MZ_a,MZ_b,MZ_R2,VaR5_violations,VaR5_rate,VaR5_expected_rate,VaR5_kupiec_LR,VaR5_kupiec_p,VaR1_violations,VaR1_rate,VaR1_expected_rate,VaR1_kupiec_LR,VaR1_kupiec_p
B1,1258.0,0.886737,0.633361,1.089793,0.164503,0.900169,0.304225,99.0,0.078696,0.05,18.7092,0.000015,20.0,0.015898,0.01,3.749256,0.052831


**Интерпретация B1:**

* α ≈ 0.18, β ≈ 0.81; persistence α+β ≈ 0.99 — типично для GARCH на индексах: волатильность очень «вязкая», шоки рассасываются медленно.
* ν ≈ 4.9 — t-распределение с тяжёлыми хвостами (как и должно быть для дневных доходностей).
* На test (2020–2024) **доля нарушений VaR(5%) = 7.9%** против ожидаемых 5%; Kupiec-LR p ≈ 1.5e-5 — модель **недооценивает хвостовой риск** в стрессовый период (COVID + 2022). Это важная содержательная находка: GARCH, обученный на «спокойном» 2010-е, не «знает» про 2020 март.